# Data Cleanser 

Load dataset

In [94]:
import numpy as np
import pandas as pd 


In [95]:
df=pd.read_csv(r"C:\Users\Lenovo\Desktop\Data PP & FE\Data Cleanser\patient_health_records_synthetic.csv")
df.columns= df.columns.str.strip()

In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Patient_id      1000 non-null   int64  
 1   Age             980 non-null    float64
 2   Gender          980 non-null    object 
 3   Region          980 non-null    object 
 4   BMI             970 non-null    float64
 5   Blood_pressure  1000 non-null   float64
 6   Cholesterol     970 non-null    float64
 7   Glucose         970 non-null    float64
 8   Disease_risk    1000 non-null   int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 70.4+ KB


In [97]:
df.isnull().sum()

Patient_id         0
Age               20
Gender            20
Region            20
BMI               30
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
dtype: int64

# Handling Missing Values

* Summary report

In [98]:
mis_rep= pd.DataFrame({'missing valuse':df.isnull().sum(),'miss_per(%)':(df.isnull().sum()/len(df))*100})
mis_rep

,missing valuse,miss_per(%)
Patient_id,0,0.0
Age,20,2.0
Gender,20,2.0
Region,20,2.0
BMI,30,3.0
Blood_pressure,0,0.0
Cholesterol,30,3.0
Glucose,30,3.0
Disease_risk,0,0.0


* Apply imputation techniques 

Simple Imputer(Numerical):Replace miissing BMI with mean or median. 

In [99]:
from sklearn.impute import SimpleImputer
print('Befor the imputation')
print(df.isnull().sum())

sm=SimpleImputer(strategy="median")
df[["BMI"]]=sm.fit_transform(df[["BMI"]])
data=pd.DataFrame(df)

print('After the imputation')
print(data.isnull().sum())

Befor the imputation
Patient_id         0
Age               20
Gender            20
Region            20
BMI               30
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
dtype: int64
After the imputation
Patient_id         0
Age               20
Gender            20
Region            20
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
dtype: int64


Simple Imputer(Categorical):Replace miissing Region with the most frequent value. 

In [100]:
print("befor imputation",df.isnull().sum())

si=SimpleImputer(strategy="most_frequent")
df[["Region"]]=si.fit_transform(df[["Region"]])
print("After imputation",df.Region.isnull().sum())
print(df.isnull().sum())

befor imputation Patient_id         0
Age               20
Gender            20
Region            20
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
dtype: int64
After imputation 0
Patient_id         0
Age               20
Gender            20
Region             0
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
dtype: int64


Most Frequent Imputation:Replace missing Gender with the most common category.

In [101]:
print("befor the imputation",df.isnull().sum())
si=SimpleImputer(strategy="most_frequent")
df[["Gender"]]=si.fit_transform(df[["Gender"]])
print("after imputation",df.isnull().sum())

befor the imputation Patient_id         0
Age               20
Gender            20
Region             0
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
dtype: int64
after imputation Patient_id         0
Age               20
Gender             0
Region             0
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
dtype: int64


Missing Indicator + Random Sample Imputation: Create binary indicator columns for missingness and use random sampling to impute values 

In [102]:

print("befor imputation",df.isnull().sum())
df['miss_Age']= df['Age'].isnull().astype(int)

r_s=df['Age'].dropna().sample(df['Age'].isnull().sum(),random_state=42)
r_s.index= df[df['Age'].isnull()].index
df.loc[df['Age'].isnull(),"Age"]=r_s
df['Age'].isnull().sum()
print("After imputation",df.isnull().sum())
 
 

befor imputation Patient_id         0
Age               20
Gender             0
Region             0
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
dtype: int64
After imputation Patient_id         0
Age                0
Gender             0
Region             0
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
miss_Age           0
dtype: int64


KNN Imputer: Apply k-Nearest Neighbors imputation for impute values

In [103]:
from sklearn.impute import KNNImputer
print("Befor the imputation",df.isnull().sum())
df.columns.tolist()
cols=['Patient_id','Age','BMI','Blood_pressure','Cholesterol','Glucose']
imp=KNNImputer(n_neighbors=5,weights='distance')      
df['Glucose']= imp.fit_transform(df[cols])[:,cols.index('Glucose')]
print('After the imputation',df.isnull().sum())


Befor the imputation Patient_id         0
Age                0
Gender             0
Region             0
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose           30
Disease_risk       0
miss_Age           0
dtype: int64
After the imputation Patient_id         0
Age                0
Gender             0
Region             0
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose            0
Disease_risk       0
miss_Age           0
dtype: int64


MIICE Algorithm: Perform chained equations imputation for multiple variables simultaneously.

In [104]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
print('****Befor imputation****')
print(df.isnull().sum())
cols2=['Blood_pressure','Cholesterol','Glucose']
im=IterativeImputer(max_iter=10 ,random_state=0)
df['Cholesterol']=im.fit_transform(df[cols])[:,cols2.index('Cholesterol')]
print('****After imputation****')
print(df.isnull().sum())

****Befor imputation****
Patient_id         0
Age                0
Gender             0
Region             0
BMI                0
Blood_pressure     0
Cholesterol       30
Glucose            0
Disease_risk       0
miss_Age           0
dtype: int64
****After imputation****
Patient_id        0
Age               0
Gender            0
Region            0
BMI               0
Blood_pressure    0
Cholesterol       0
Glucose           0
Disease_risk      0
miss_Age          0
dtype: int64


# Handling Outliers

* Detect and remove outliers

Z-score method: Identify patients with extreme cholestrol or glucose values.

In [105]:
mean=df['Cholesterol'].mean()
Std=df['Cholesterol'].std()
df['ZC']=(df['Cholesterol']-mean)/Std

dfz=df[df['ZC'].abs()<=3].drop(columns= ['ZC'])
df=df.drop(columns=['ZC'])
print('shape with outlier',df.shape)
print('mean with outlier',df['Cholesterol'].mean())
print('shape without outlier',dfz.shape)
print('mean without outlier',dfz['Cholesterol'].mean())

shape with outlier (1000, 10)
mean with outlier 47.738
shape without outlier (1000, 10)
mean without outlier 47.738


IQR method: use interquartile range to detect unusual BMI values.

In [106]:
q1=df['BMI'].quantile(0.25)
q3=df['BMI'].quantile(0.75)
iqr=q3-q1
upl=q3+1.5*iqr
lwl=q1-1.5*iqr

outlier=(df['BMI']>upl) | (df['BMI']<lwl)   
df_iq=df[~outlier]

print('shape with outlier',df.shape)
print('mean with outlier',df['BMI'].mean())
print('shape without outlier',df_iq.shape)
print('mean without outlier',df_iq['BMI'].mean())

shape with outlier (1000, 10)
mean with outlier 27.1662
shape without outlier (995, 10)
mean without outlier 27.11396984924623


Perecentile method: Cap values below 1st percentile and above 99th percentile.

In [107]:
up_limit=df['Glucose'].quantile(0.95)
lo_limit=df['Glucose'].quantile(0.05)
mask=(df['Glucose']<lo_limit) | (df['Glucose']>up_limit)
df_p=df[~mask]
print('shape with outlier',df.shape)
print('mean with outlier',df['Glucose'].mean())
print('shape without outlier',df_p.shape)
print('mean without outlier',df_p['Glucose'].mean())

shape with outlier (1000, 10)
mean with outlier 105.2399137110988
shape without outlier (900, 10)
mean without outlier 104.70657079010977


* Apply Winsorization to cap extreme outliers instead of removing them.

In [108]:
df['Cholesterol_w']=df['Cholesterol'].clip(lower=lo_limit,upper=up_limit)

print('shape with outlier',df.shape)
print('mean with outlier',df['Cholesterol'].mean())
print('shape without outlier',df.shape)
print('mean without outlier',df['Cholesterol_w'].mean())

shape with outlier (1000, 11)
mean with outlier 47.738
shape without outlier (1000, 11)
mean without outlier 63.32625


* Compare dataset shape and summary before vs after outlier treatment 

In [109]:
data={'Method':['Z-score(Cholesterol)','IQR(BMI)','Perecentile(Glucose)','Winsorization(Cholesterol)'],
      'mean_with outlier':[df['Cholesterol'].mean(),df['BMI'].mean(),df['Glucose'].mean(),df['Cholesterol'].mean()],
      'mean without outlier':[dfz['Cholesterol'].mean(),df_iq['BMI'].mean(),df_p['Glucose'].mean(),df['Cholesterol_w'].mean()],
      'STD_with outlier':[df['Cholesterol'].std(),df['BMI'].std(),df['Glucose'].std(),df['Cholesterol'].std()],
      'STD_without outlier':[dfz['Cholesterol'].std(),df_iq['BMI'].std(),df_p['Glucose'].std(),df['Cholesterol_w'].std()]}

com=pd.DataFrame(data)
com

,Method,mean_with outlier,mean without outlier,STD_with outlier,STD_without outlier
0,Z-score(Cholesterol),47.738000,47.738000,15.289838,15.289838
1,IQR(BMI),27.166200,27.113970,5.122636,4.969423
2,Perecentile(Glucose),105.239914,104.706571,25.375801,20.000844
3,Winsorization(Cholesterol),47.738000,63.326250,15.289838,4.488055


# Final clean Dataset 

* Present the final cleaned dataset with  

In [110]:
df
df.isnull().sum()
final_data=df
final_data.to_csv('patient_health_clean_dataset.csv',index=False)   
print(final_data.isnull().sum())

Patient_id        0
Age               0
Gender            0
Region            0
BMI               0
Blood_pressure    0
Cholesterol       0
Glucose           0
Disease_risk      0
miss_Age          0
Cholesterol_w     0
dtype: int64


All missing values treated appropriately.

All outliers handled using suitable methods.

* Provide a brief report explaining 

Which imputation strategy was effective.

:Different imputation techniques were applied to handle missing values in the patient health records dataset. Median imputation was effective for BMI because it is less sensitive to extreme values. Most Frequent Imputation successfully handled missing categorical values such as Gender and Region. Random Sample Imputation preserved the original distribution of Age values, while KNN and MICE successfully filled missing numerical values in Glucose and Cholesterol, respectively. Overall, MICE was effective for Cholesterol, while different imputation methods were suitable for different columns.

Which outlier handling method preserved data quality best.

:The IQR method performed best for BMI because it removed only 5 unusual records out of 1000, retaining 995 records. The BMI mean changed slightly from 27.1662 to 27.114, and standard deviation decreased from 5.1226 to 4.9694. This indicates that IQR reduced the influence of extreme values while preserving most of the dataset. Therefore, IQR was the most suitable method in this comparison for maintaining data quality with minimal information loss.

How data cleaning improved dataset usability

:Data cleaning improved the quality, consistency, and reliability of the patient health records dataset. Missing values were handled using suitable imputation techniques, reducing missing data across important columns. Outlier detection helped identify unusual BMI, Cholesterol, and Glucose values. The cleaned dataset contained 1000 records with zero missing values, making it more suitable for statistical analysis, visualization, and machine learning. Overall, data cleaning improved dataset usability and supported more reliable analytical results.

 